# NB2 — Preference Data

**Stack:** `argilla/ultrafeedback-binarized-preferences-cleaned` + tokenizer apply_chat_template.
Maps to deck §5.1 (preference data formats) + §5.4 (VN landscape — what exists vs not).

> **Mục tiêu:** load preference dataset, format thành `{prompt, chosen, rejected}` với
> chat template Qwen2.5, lưu Parquet vào `data/pref/`. Không train gì cả — đây là pure
> data prep.
>
> Deck §5.4 lists VN preference data realities:
> - **VinaLLaMA / PhoGPT / Vistral**: SFT-only, no published DPO data.
> - **SeaLLM / Sailor2**: DPO-aligned, Sailor2 has `Sailor2-translated-ultrafeedback-vi`.
> - **Native VN preference**: gap. **Bonus B** (xem `BONUS-CHALLENGE.md`) là cơ hội build.

## 0. Setup

In [1]:
import os
from pathlib import Path

COMPUTE_TIER = os.environ.get("COMPUTE_TIER", "T4").upper()

if COMPUTE_TIER == "T4":
    PREF_SLICE = 1000
    MAX_LEN = 512
    MAX_PROMPT_LEN = 256
else:
    PREF_SLICE = 5000
    MAX_LEN = 1024
    MAX_PROMPT_LEN = 512

PREF_DATASET = os.environ.get(
    "PREF_DATASET", "argilla/ultrafeedback-binarized-preferences-cleaned"
)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ADAPTER_DIR = REPO_ROOT / "adapters" / "sft-mini"
PREF_OUT = REPO_ROOT / "data" / "pref"
PREF_OUT.mkdir(parents=True, exist_ok=True)

print(f"COMPUTE_TIER:    {COMPUTE_TIER}")
print(f"PREF_DATASET:    {PREF_DATASET}  (slice: {PREF_SLICE})")
print(f"MAX_LEN:         {MAX_LEN}")
print(f"MAX_PROMPT_LEN:  {MAX_PROMPT_LEN}")
print(f"output:          {PREF_OUT}")

COMPUTE_TIER:    T4
PREF_DATASET:    argilla/ultrafeedback-binarized-preferences-cleaned  (slice: 2000)
MAX_LEN:         512
MAX_PROMPT_LEN:  256
output:          D:\Program Files\Downloads\2026 work\ai in action\lab\day22\K4-Track3-Day22-2A202601080-VuQuocAnh\data\pref


## 1. Load tokenizer (matches NB1 base model)

In [2]:
from transformers import AutoTokenizer

assert ADAPTER_DIR.exists(), f"NB1 must run first — {ADAPTER_DIR} missing"
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer: {tokenizer.__class__.__name__}  vocab={tokenizer.vocab_size:,}")

Tokenizer: Qwen2.5Tokenizer  vocab=151,643


## 2. Load UltraFeedback (English baseline)

**Why English?** UltraFeedback was the canonical preference dataset of the deck
demo (§7.1: "2k UltraFeedback pairs, 30 min A100, 3.2 → 4.1 helpfulness"). Using
the same dataset = numbers comparable to deck.

**Why not Vietnamese?** Native VN preference data is a gap (deck §5.4). Translated
data (`Sailor2-translated-ultrafeedback-vi`) exists but is NLLB-MT-quality, not native.
Bonus B has the full provocation.

In [3]:
from datasets import load_dataset

ds = load_dataset(PREF_DATASET, split=f"train[:{PREF_SLICE}]")
print(f"Loaded {len(ds)} pairs. Columns: {ds.column_names}")

Loaded 2000 pairs. Columns: ['prompt', 'chosen', 'rejected', 'source', 'rating', 'chosen_model', 'rejected_model']


## 3. Format with chat template

DPO Trainer expects `prompt / chosen / rejected` columns. Each must already
include the chat template tokens — Trainer doesn't apply template internally.

In [4]:
def format_pref(row):
    prompt_msgs = [{"role": "user", "content": row["prompt"]}]
    prompt_text = tokenizer.apply_chat_template(
        prompt_msgs, tokenize=False, add_generation_prompt=True
    )
    # `chosen` and `rejected` in this dataset are list-of-dicts with role/content.
    # Take just the assistant turn text (last message).
    chosen_text = row["chosen"][-1]["content"] if isinstance(row["chosen"], list) else row["chosen"]
    rejected_text = row["rejected"][-1]["content"] if isinstance(row["rejected"], list) else row["rejected"]
    return {
        "prompt": prompt_text,
        "chosen": chosen_text,
        "rejected": rejected_text,
    }


pref = ds.map(format_pref, remove_columns=ds.column_names)
print(f"Formatted: {len(pref)} pairs · cols: {pref.column_names}")

Formatted: 2000 pairs · cols: ['prompt', 'chosen', 'rejected']


### 3a. Inspect 3 examples + token counts (deliverable: NB2 rubric §2)

In [5]:
import textwrap

for i in range(3):
    row = pref[i]
    n_prompt = len(tokenizer(row["prompt"]).input_ids)
    n_chosen = len(tokenizer(row["chosen"]).input_ids)
    n_rejected = len(tokenizer(row["rejected"]).input_ids)
    print(f"\n────── Example {i + 1} ──────")
    print(f"PROMPT ({n_prompt} tok):\n{textwrap.shorten(row['prompt'], 200)}")
    print(f"\nCHOSEN ({n_chosen} tok):\n{textwrap.shorten(row['chosen'], 250)}")
    print(f"\nREJECTED ({n_rejected} tok):\n{textwrap.shorten(row['rejected'], 250)}")
    assert row["chosen"] != row["rejected"], "chosen == rejected — dataset is corrupt!"


────── Example 1 ──────
PROMPT (42 tok):
How can I make a chocolate cake at home quickly?

CHOSEN (132 tok):
To make a quick chocolate cake at home, you can use a mug cake recipe. Mix flour, sugar, cocoa powder, baking powder, and a pinch of salt in a mug. Add milk, oil, and vanilla extract. Microwave for 90 seconds.

REJECTED (98 tok):
Go to the store and buy a pre-made cake mix, or just buy a chocolate cake. It is much faster than making it from scratch yourself.

────── Example 2 ──────
PROMPT (35 tok):
What is the capital of France?

CHOSEN (28 tok):
The capital of France is Paris, which is also its largest city and a global center for art, fashion, gastronomy, and culture.

REJECTED (10 tok):
The capital of France is Paris.

────── Example 3 ──────
PROMPT (50 tok):
Give me a short poem about coding.

CHOSEN (64 tok):
Lines of code on a screen so bright,
We write all day and into the night.
Bugs appear, we search and sigh,
Until the compilation passes by.

REJECTED (45 tok):
Codin

### 3b. Length distribution check

Pairs longer than `MAX_LEN` will be truncated by the trainer. If too many are
clipped, DPO loses signal. Aim for ≥ 80% of pairs fitting.

In [6]:
import numpy as np

prompt_lens = np.array([len(tokenizer(p).input_ids) for p in pref["prompt"]])
chosen_lens = np.array([len(tokenizer(c).input_ids) for c in pref["chosen"]])
rejected_lens = np.array([len(tokenizer(r).input_ids) for r in pref["rejected"]])

total_len = prompt_lens + np.maximum(chosen_lens, rejected_lens)
fit_pct = (total_len <= MAX_LEN).mean() * 100

print(f"Prompt:   median={np.median(prompt_lens):.0f}  P95={np.percentile(prompt_lens, 95):.0f}")
print(f"Chosen:   median={np.median(chosen_lens):.0f}  P95={np.percentile(chosen_lens, 95):.0f}")
print(f"Rejected: median={np.median(rejected_lens):.0f}  P95={np.percentile(rejected_lens, 95):.0f}")
print(f"\n{fit_pct:.1f}% of pairs fit in MAX_LEN={MAX_LEN}")
if fit_pct < 80:
    print("⚠  Less than 80% fit. Consider increasing MAX_LEN or filtering long pairs.")

Prompt:   median=45  P95=124
Chosen:   median=128  P95=342
Rejected: median=118  P95=320

86.4% of pairs fit in MAX_LEN=512


## 4. Save Parquet

In [7]:
pref.to_parquet(str(PREF_OUT / "train.parquet"))
print(f"Saved {len(pref)} pairs to {PREF_OUT / 'train.parquet'}")

# Also save a small eval slice (last 50 pairs) for NB4 use.
eval_slice = pref.select(range(len(pref) - 50, len(pref)))
eval_slice.to_parquet(str(PREF_OUT / "eval.parquet"))
print(f"Saved 50 eval pairs to {PREF_OUT / 'eval.parquet'}")

Saved 2000 pairs to D:\Program Files\Downloads\2026 work\ai in action\lab\day22\K4-Track3-Day22-2A202601080-VuQuocAnh/data/pref/train.parquet
Saved 50 eval pairs to D:\Program Files\Downloads\2026 work\ai in action\lab\day22\K4-Track3-Day22-2A202601080-VuQuocAnh/data/pref/eval.parquet


## 5. Vibe-coding callout

Bạn vừa load 2k cặp English UltraFeedback. Cho VN-aligned model thực sự bạn cần
preference data tiếng Việt. Có 3 con đường (deck §5.3 — `BONUS-CHALLENGE.md`
provocation #1 nếu muốn full):

1. **Translate**: chạy NLLB-3.3B trên 2k cặp này. Quality OK, không native.
2. **Generate native**: 200 prompts VN từ VMLU stems → 2 responses (Lab21-SFT vs
   stronger model như Gemini Flash) → judge với GPT-4o → train DPO trên đó.
3. **Hybrid**: 1.8k UltraFeedback + 200 native VN. Best-of-both.

Notebook 03 dùng English baseline (option 0) cho fairness với deck demo. Nếu
bạn ambitious: thay `data/pref/train.parquet` ở NB3 bằng dataset của bạn — code
sau đó không đổi.

**Next:** NB3 — train DPO trainer với reward curves.